In [1]:
import os
import re
import json
from nltk.stem import PorterStemmer

class BooleanRetrievalModel:
    def __init__(self, data_dir, stopwords_file):
        """
        Initializes the IR model, loads stopwords, and sets up the stemmer[cite: 30, 31, 517].
        """
        self.data_dir = data_dir
        self.stemmer = PorterStemmer()
        self.stopwords = self._load_stopwords(stopwords_file)

        # Data structures for indexes [cite: 6, 28, 611]
        self.inverted_index = {}
        self.positional_index = {}
        self.all_docs = set()

    def _load_stopwords(self, filepath):
        """Loads the stop words list[cite: 39, 53]."""
        stopwords = set()
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                for line in f:
                    stopwords.add(line.strip().lower())
        return stopwords

    def preprocess(self, text):
        """
        Pre-processing pipeline: Tokenization, case folding,
        stop-words removal, and stemming[cite: 30, 489, 517].
        """
        # Case folding and Tokenization
        tokens = re.findall(r'\b\w+\b', text.lower())

        processed_tokens = []
        for token in tokens:
            # Stop-words removal
            if token not in self.stopwords:
                # Stemming using Porter Stemmer [cite: 524]
                stemmed_token = self.stemmer.stem(token)
                processed_tokens.append(stemmed_token)

        return processed_tokens

    def build_indexes(self):
        """
        Reads the 56 files, treats each as a unique document,
        and builds both Inverted and Positional Indexes[cite: 9, 10, 12].
        """
        print("Building indexes...")
        if not os.path.exists(self.data_dir):
            print(f"Error: Directory '{self.data_dir}' not found.")
            return

        for filename in os.listdir(self.data_dir):
            if filename.endswith(".txt"):
                doc_id = filename
                self.all_docs.add(doc_id)
                filepath = os.path.join(self.data_dir, filename)

                with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()

                tokens = self.preprocess(text)

                for position, term in enumerate(tokens):
                    # --- Build Inverted Index [cite: 280] ---
                    if term not in self.inverted_index:
                        self.inverted_index[term] = set()
                    self.inverted_index[term].add(doc_id)

                    # --- Build Positional Index [cite: 610-612] ---
                    if term not in self.positional_index:
                        self.positional_index[term] = {}
                    if doc_id not in self.positional_index[term]:
                        self.positional_index[term][doc_id] = []
                    self.positional_index[term][doc_id].append(position)

        # Convert sets to sorted lists for standard algorithm compatibility
        for term in self.inverted_index:
            self.inverted_index[term] = sorted(list(self.inverted_index[term]))

        print(f"Indexes built successfully for {len(self.all_docs)} documents.")

    def save_indexes(self, inv_file='inverted_index.json', pos_file='positional_index.json'):
        """Saves indexes to disk (1 mark for saving/loading)[cite: 45, 46]."""
        with open(inv_file, 'w') as f:
            json.dump(self.inverted_index, f)
        with open(pos_file, 'w') as f:
            json.dump(self.positional_index, f)
        print("Indexes saved to disk.")

    def load_indexes(self, inv_file='inverted_index.json', pos_file='positional_index.json'):
        """Loads indexes from disk[cite: 45, 46]."""
        if os.path.exists(inv_file) and os.path.exists(pos_file):
            with open(inv_file, 'r') as f:
                self.inverted_index = json.load(f)
            with open(pos_file, 'r') as f:
                self.positional_index = json.load(f)
            print("Indexes loaded from disk.")
            return True
        return False

    # --- Slide Algorithms ---

    def intersect(self, p1, p2):
        """
        Standard Intersect Algorithm from Week 1b Slides[cite: 301].
        """
        answer = []
        i, j = 0, 0
        while i < len(p1) and j < len(p2):
            if p1[i] == p2[j]:
                answer.append(p1[i])
                i += 1
                j += 1
            elif p1[i] < p2[j]:
                i += 1
            else:
                j += 1
        return answer

    def positional_intersect(self, p1_dict, p2_dict, k):
        """
        Positional Intersect Algorithm from Week 2b Slides [cite: 641-690].
        Adapted for Python dictionary structures.
        """
        answer = []
        # Get sorted doc IDs that exist in both terms
        doc_ids1 = sorted(p1_dict.keys())
        doc_ids2 = sorted(p2_dict.keys())

        i, j = 0, 0
        while i < len(doc_ids1) and j < len(doc_ids2):
            doc1, doc2 = doc_ids1[i], doc_ids2[j]

            if doc1 == doc2:
                l = []
                pp1 = p1_dict[doc1] # positions of term 1
                pp2 = p2_dict[doc2] # positions of term 2

                ii, jj = 0, 0
                while ii < len(pp1):
                    while jj < len(pp2):
                        if abs(pp1[ii] - pp2[jj]) <= k:
                            l.append(pp2[jj])
                        elif pp2[jj] > pp1[ii]:
                            break
                        jj += 1

                    while len(l) > 0 and abs(l[0] - pp1[ii]) > k:
                        l.pop(0)

                    for ps in l:
                        answer.append(doc1) # We just need the doc ID for the final result
                    ii += 1

                i += 1
                j += 1
            elif doc1 < doc2:
                i += 1
            else:
                j += 1

        # Return unique matching document IDs
        return sorted(list(set(answer)))

    # --- Query Processing ---

    def process_query(self, query):
        """
        Processes Boolean combinations (max 3 terms) and Proximity queries[cite: 22, 27].
        """
        # Handle Proximity Query first (e.g., term1 term2 /3) [cite: 27]
        if '/' in query:
            parts = query.split('/')
            k = int(parts[1].strip())
            terms = self.preprocess(parts[0])

            if len(terms) == 2:
                t1, t2 = terms[0], terms[1]
                p1_dict = self.positional_index.get(t1, {})
                p2_dict = self.positional_index.get(t2, {})
                return self.positional_intersect(p1_dict, p2_dict, k)
            else:
                return "Proximity query requires exactly two terms."

        # Handle Standard Boolean Query (AND, OR, NOT) [cite: 23]
        tokens = query.split()

        # Simple parser for max 3 terms and operators
        result_docs = set(self.all_docs) # Start with all docs for clean logical NOTs
        current_set = set()
        operator = "OR" # Default start operator

        i = 0
        while i < len(tokens):
            word = tokens[i]
            if word in ["AND", "OR", "NOT"]:
                operator = word
                i += 1
                continue

            # Preprocess the query term
            stemmed_term = self.preprocess(word)
            if not stemmed_term:
                i += 1
                continue
            term = stemmed_term[0]

            # Fetch postings list
            term_docs = set(self.inverted_index.get(term, []))

            if operator == "AND":
                # Convert back to sorted lists to use the slide's INTERSECT algorithm [cite: 301]
                current_set = set(self.intersect(sorted(list(current_set)), sorted(list(term_docs))))
            elif operator == "OR":
                current_set = current_set.union(term_docs)
            elif operator == "NOT":
                # doc that do not contain X [cite: 26]
                not_docs = set(self.all_docs) - term_docs
                if i == 1: # If query starts with NOT X
                    current_set = not_docs
                else:
                    current_set = set(self.intersect(sorted(list(current_set)), sorted(list(not_docs))))

            # Initialization case for the very first term
            if i == 0:
                current_set = term_docs

            i += 1

        return sorted(list(current_set))

# --- Command Line Interface ---

if __name__ == "__main__":
    # Ensure you have extracted your files into a folder named 'Trump_Speeches'
    DATA_DIRECTORY = "Trump_Speeches"
    STOPWORDS_FILE = "Stopword-List.txt"

    ir_system = BooleanRetrievalModel(DATA_DIRECTORY, STOPWORDS_FILE)

    # Try loading existing indexes to save time, otherwise build them [cite: 45]
    if not ir_system.load_indexes():
        ir_system.build_indexes()
        ir_system.save_indexes()

    print("\n--- Boolean Information Retrieval Model ---")
    print("Supports Boolean Operators: AND, OR, NOT (Max 3 terms) [cite: 22, 23]")
    print("Supports Proximity Queries: term1 term2 /k (e.g., great wall /3) [cite: 27]")
    print("Type 'exit' to quit.\n")

    while True:
        user_query = input("Enter your query: ")
        if user_query.lower() == 'exit':
            break

        results = ir_system.process_query(user_query)

        if isinstance(results, str):
            print(f"Error: {results}")
        else:
            print(f"Documents retrieved ({len(results)}): {results}\n")

ModuleNotFoundError: No module named 'nltk'